In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# =============================================================================
# Notebook  : 02b_map_02_accounts_all
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_02_accounts_all
# Spec Ref  : §1.2 Refresh Materialized Views
# Purpose   : Build accounts_all Delta table = accounts UNION ALL crm_accounts
#             (currently just hgi.silver.accounts — ready for multi-source union)
#             Run AFTER map_01 (contacts_all must exist first).
#
# Serverless: works on 2X-Small
# Job Compute: full perf configs applied
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:

%run ./_map_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2 ── Widget
dbutils.widgets.text("customer_id", "cust_005")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
tenant_id   = tenant_id_from_customer(customer_id)

print(f"=== Map 02: accounts_all ===  customer={customer_id}  tenant={tenant_id}")
print(f"  Reading from : {sv}.accounts")
print(f"  Writing to   : {sv}.accounts_all")

In [0]:
source_system = "salesforce"
object_name   = "map"
load_type     = "incremental"
initialize_audit_tables()

In [0]:
# CELL 3 ── CDF-Aware Incremental MERGE for accounts_all (tenant-scoped)
# FIX v2: Use SOURCE table version (stored as property) instead of TARGET history
# This prevents version mismatch errors when target has more versions than source
try:
    safe_drop_view(f"{sv}.accounts_all")

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {sv}.accounts_all (
            id                    STRING NOT NULL,
            tenant                BIGINT,
            domain                STRING,
            name                  STRING,
            source_system         STRING,
            source_system_object  STRING,
            source_key_name       STRING,
            source_key_value      STRING,
            data_timestamp        TIMESTAMP
        )
        USING DELTA
        CLUSTER BY (tenant, domain)
        {DELTA_TBLPROPS_MAP}
    """)

    # ── VERSION TRACKING (v2): Read last-processed SOURCE version from target property ──
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {sv}.accounts_all ('last_source_cdf_version')").collect()
        val = props[0][1] if props else None
        last_source_ver = int(val) if val and val != '(not specified)' else 0
    except Exception:
        last_source_ver = 0

    # Get current max version of the SOURCE table (silver.accounts)
    try:
        source_max_ver = spark.sql(f"SELECT MAX(version) FROM (DESCRIBE HISTORY {sv}.accounts)").collect()[0][0] or 0
    except Exception:
        source_max_ver = 0

    print(f"  CDF: source={sv}.accounts  last_read_ver={last_source_ver}  current_source_ver={source_max_ver}")

    # Check if this is first run (accounts_all is empty) — do full load
    target_count = spark.sql(f"SELECT COUNT(*) FROM {sv}.accounts_all WHERE tenant = {tenant_id}").collect()[0][0]
    
    if target_count == 0:
        # First run: full load from source
        print(f"  First run detected — loading all accounts for tenant {tenant_id}")
        source_df = spark.sql(f"""
            SELECT id, tenant, domain, name, source_system, source_system_object,
                   source_key_name, source_key_value, data_timestamp
            FROM {sv}.accounts
            WHERE id IS NOT NULL AND tenant = {tenant_id}
        """)
        cdf_count = source_df.count()
    elif last_source_ver >= source_max_ver:
        # No new versions on source — nothing to process
        print(f"  Source table unchanged (ver {source_max_ver}) — skipping")
        cdf_count = 0
        source_df = None
    else:
        # Incremental: read only CDF changes from silver.accounts
        start_ver = last_source_ver + 1
        try:
            cdf_df = (spark.read
                .format("delta")
                .option("readChangeFeed", "true")
                .option("startingVersion", start_ver)
                .table(f"{sv}.accounts")
                .filter(f"tenant = {tenant_id}")
                .filter("_change_type IN ('insert', 'update_postimage')")
            )
            cdf_count = cdf_df.count()
            print(f"  CDF records found: {cdf_count} (versions {start_ver}..{source_max_ver})")
            
            if cdf_count == 0:
                print(f"  No changes detected — skipping MERGE")
                source_df = None
            else:
                source_df = cdf_df.select(
                    "id", "tenant", "domain", "name", "source_system", "source_system_object",
                    "source_key_name", "source_key_value", "data_timestamp"
                )
        except Exception as cdf_err:
            # CDF not available or error — fall back to watermark-based filtering
            print(f"  CDF read failed ({cdf_err}), falling back to watermark-based filter")
            last_ts = spark.sql(f"""
                SELECT COALESCE(MAX(data_timestamp), TIMESTAMP('1900-01-01'))
                FROM {sv}.accounts_all WHERE tenant = {tenant_id}
            """).collect()[0][0]
            source_df = spark.sql(f"""
                SELECT id, tenant, domain, name, source_system, source_system_object,
                       source_key_name, source_key_value, data_timestamp
                FROM {sv}.accounts
                WHERE id IS NOT NULL AND tenant = {tenant_id}
                  AND data_timestamp > '{last_ts}'
            """)
            cdf_count = source_df.count()
            print(f"  Watermark-based records found: {cdf_count}")

    # Only run MERGE if we have changes
    if source_df is not None and cdf_count > 0:
        source_df.createOrReplaceTempView("accounts_cdf_source")
        
        spark.sql(f"""
            MERGE INTO {sv}.accounts_all AS target
            USING accounts_cdf_source AS source
            ON target.id = source.id AND target.tenant = source.tenant
            WHEN MATCHED AND source.data_timestamp > target.data_timestamp THEN UPDATE SET
                target.domain = source.domain,
                target.name = source.name,
                target.source_system = source.source_system,
                target.source_system_object = source.source_system_object,
                target.source_key_name = source.source_key_name,
                target.source_key_value = source.source_key_value,
                target.data_timestamp = source.data_timestamp
            WHEN NOT MATCHED THEN INSERT (id, tenant, domain, name, source_system,
                source_system_object, source_key_name, source_key_value, data_timestamp)
                VALUES (source.id, source.tenant, source.domain, source.name, source.source_system,
                    source.source_system_object, source.source_key_name, source.source_key_value,
                    source.data_timestamp)
        """)
        print(f"  MERGE complete: {cdf_count} CDF records processed")
    else:
        print(f"  No changes to process")

    # ── Save the source version we just processed ──
    spark.sql(f"ALTER TABLE {sv}.accounts_all SET TBLPROPERTIES ('last_source_cdf_version' = '{source_max_ver}')")
    print(f"  Saved last_source_cdf_version = {source_max_ver}")

except Exception as e:
    print(f"❌ accounts_all build failed: {e}")
    log_audit(
        job_name       = "02b_map_02_accounts_all",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    raise

In [0]:
# CELL 4 ── Verify
try:
    n = cnt(f"{sv}.accounts_all")
    print(f"  accounts_all: {n:,} rows built")

    # Spec DQ gate: domain must be lowercase
    bad_domain = spark.sql(f"""
        SELECT COUNT(*) FROM {sv}.accounts_all
        WHERE domain IS NOT NULL AND domain != LOWER(domain)
    """).collect()[0][0]
    if bad_domain > 0:
        print(f"  WARNING: {bad_domain} accounts with uppercase domain")
    else:
        print(f"  DQ OK: all domains lowercase")
except Exception as e:
    print(f"❌ DQ verification failed: {e}")
    log_audit(
        job_name       = "02b_map_02_accounts_all",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "DQ_CHECK_FAILED",
        error_type     = type(e).__name__,
        error_reason   = f"DQ check failed: {str(e)[:450]}",
    )
    raise